# Capítulo 8 · Laboratório 8: indicadores escritos à mão
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap08.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Todos os indicadores estão escritos à mão, com pandas, para você ver cada conta. O laboratório refaz a história de John Bollinger (nove indicadores de momentum valem 1,6 opiniões), mostra um indicador de cada família, prova que o Williams %R é o estocástico invertido, testa os indicadores em dados inventados e mostra o aquecimento e o sobreajuste do IFR.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap08.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
SERIES = {"PETR4.SA": "2010-01-01", "VALE3.SA": "2010-01-01", "ITUB4.SA": "2010-01-01", "^BVSP": "2005-01-01"}


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t, ini in SERIES.items():
    if t not in D:
        D[t] = baixa(t, ini)
confere({t: D[t] for t in SERIES})
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d, indice=False):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if not indice:
        ok &= d["Volume"] > 0
    return d[ok].copy()


PETR, VALE, ITUB = (limpa(D[t]) for t in ("PETR4.SA", "VALE3.SA", "ITUB4.SA"))
IBOV = limpa(D["^BVSP"], True)
print(f"Parte 1. PETR4, VALE3 e ITUB4 desde {PETR.index[0]:%Y} (preços ajustados); Ibovespa desde {IBOV.index[0]:%Y};"
      f" último pregão {IBOV.index[-1]:%d/%m/%Y}.")

## Parte 2: os indicadores, escritos à mão

In [ ]:
def ifr(c, n=14):
    """IFR de Wilder: média (suavização de Wilder) dos ganhos dividida pela das perdas."""
    delta = c.diff()
    ganho = delta.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    perda = (-delta.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + ganho / perda)


def estocastico(d, n=14):
    """%K de Lane: onde o fechamento está dentro da faixa máxima-mínima dos últimos n pregões (0 a 100)."""
    return 100 * (d["Close"] - d["Low"].rolling(n).min()) / (d["High"].rolling(n).max() - d["Low"].rolling(n).min())


def williams(d, n=14):
    """%R de Larry Williams: a mesma conta medida a partir da máxima (0 a -100)."""
    return -100 * (d["High"].rolling(n).max() - d["Close"]) / (d["High"].rolling(n).max() - d["Low"].rolling(n).min())


def cci(d, n=20):
    tp = (d["High"] + d["Low"] + d["Close"]) / 3
    desvio = tp.rolling(n).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
    return (tp - tp.rolling(n).mean()) / (0.015 * desvio)


def macd(c, r=12, l=26):
    return c.ewm(span=r, adjust=False).mean() - c.ewm(span=l, adjust=False).mean()


def mfi(d, n=14):
    tp = (d["High"] + d["Low"] + d["Close"]) / 3
    fluxo = tp * d["Volume"]
    pos = fluxo.where(tp > tp.shift(), 0).rolling(n).sum()
    neg = fluxo.where(tp < tp.shift(), 0).rolling(n).sum()
    return 100 - 100 / (1 + pos / neg)


def atr(d, n=14):
    tr = pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift()).abs(),
                    (d["Low"] - d["Close"].shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def obv(d):
    return (np.sign(d["Close"].diff()).fillna(0) * d["Volume"]).cumsum()


def opinioes(m):
    """Quantas opiniões independentes a tabela de correlação realmente tem (razão de participação dos
    autovalores): vai de 1 (todos dizem a mesma coisa) até o número de indicadores (todos independentes)."""
    lam = np.linalg.eigvalsh(m.values)
    return lam.sum() ** 2 / (lam ** 2).sum()


fora_diag = lambda m: m.values[~np.eye(len(m), dtype=bool)].mean()

## Parte 3: nove indicadores de momentum

In [ ]:
def nove(d):
    c = d["Close"]
    return pd.DataFrame({"IFR": ifr(c), "Estocástico": estocastico(d), "Williams %R": williams(d), "CCI": cci(d),
                         "ROC 10": c.pct_change(10) * 100, "Momento 10": c - c.shift(10), "MACD": macd(c),
                         "PPO": macd(c) / c.ewm(span=26, adjust=False).mean() * 100, "MFI": mfi(d)}).dropna()


M = nove(PETR.loc["2019":])
CM = M.corr(method="spearman")
print("\nParte 3. Nove indicadores de momentum em PETR4 (desde 2019; correlação de postos)")
print(CM.round(2).to_string())
OPN = {}
for nome, d in (("PETR4", PETR), ("VALE3", VALE), ("ITUB4", ITUB)):
    cm = nove(d.loc["2019":]).corr(method="spearman")
    OPN[nome] = (fora_diag(cm), opinioes(cm))
    print(f"   {nome}: correlação média entre pares {nr(OPN[nome][0])}; valem {nr(OPN[nome][1], 1)} opiniões"
          f" independentes, e não 9")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(CM.values, cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(range(len(CM)))
ax.set_xticklabels(CM.columns, rotation=45, ha="right")
ax.set_yticks(range(len(CM)))
ax.set_yticklabels(CM.columns)
ax.set_title("Quanto mais escuro, mais os dois indicadores dizem a mesma coisa")
plt.show()

## Parte 4: um de cada família

In [ ]:
def familias(d):
    c = d["Close"]
    o = obv(d)
    return pd.DataFrame({
        "Tendência: distância da MM200 (%)": (c / c.rolling(200).mean() - 1) * 100,
        "Momentum: IFR(14)": ifr(c),
        "Volume: OBV, variação de 20 dias (z)": o.diff(20) / o.diff(20).rolling(250).std(),
        "Volatilidade: ATR em % do preço": atr(d) / c * 100}).dropna()


F = familias(PETR.loc["2018":])
CF = F.corr(method="spearman")
print("\nParte 4. Um indicador de cada família (PETR4)")
print(CF.round(2).to_string())
print(f"   correlação média entre pares {nr(fora_diag(CF))}; valem {nr(opinioes(CF), 1)} opiniões independentes de 4")

## Parte 5: estocástico e Williams %R

In [ ]:
K, R = estocastico(PETR), williams(PETR)
ok = K.notna() & R.notna()
print(f"\nParte 5. Estocástico x Williams %R em PETR4: correlação {np.corrcoef(K[ok], R[ok])[0, 1]:.4f};"
      f" maior diferença entre %R e (%K - 100): {float((R[ok] - (K[ok] - 100)).abs().max()):.2e}")

## Parte 6: limitados e ilimitados

In [ ]:
print("\nParte 6. Limitados (IFR) e ilimitados (MACD) em 30/09/2026:")
LIM = {}
for nome, d in (("PETR4", PETR), ("VALE3", VALE), ("ITUB4", ITUB), ("Ibovespa", IBOV)):
    c = d["Close"]
    m = macd(c)
    LIM[nome] = (ifr(c).iloc[-1], m.iloc[-1], m.loc["2016":].abs().max())
    print(f"   {nome:<9} IFR(14) = {nr(LIM[nome][0], 1):>5}; MACD = {nr(LIM[nome][1]):>10};"
          f" maior MACD em módulo desde 2016 = {nr(LIM[nome][2])}")

## Parte 7: dados sintéticos

In [ ]:
dias = np.arange(160)
preco = np.where(dias < 60, 100 - 0.5 * dias, 70 + 0.5 * (dias - 60))       # fundo em V no dia 60
SINT = pd.DataFrame({"Close": preco, "High": preco + 0.5, "Low": preco - 0.5, "Volume": 1.0})
SINAIS = {"Estocástico(14) cruza 50": estocastico(SINT) > 50,
          "Preço cruza MM20": SINT["Close"] > SINT["Close"].rolling(20).mean(),
          "IFR(14) cruza 50": ifr(SINT["Close"]) > 50,
          "MACD(12,26) cruza zero": macd(SINT["Close"]) > 0,
          "Preço cruza MM50": SINT["Close"] > SINT["Close"].rolling(50).mean()}
ATRASO = {}
print("\nParte 7. Série sintética com fundo em V no dia 60: atraso de cada sinal de alta")
for nome, cond in SINAIS.items():
    dia = int(cond[cond & (cond.index > 60)].index.min())
    ATRASO[nome] = dia - 60
    print(f"   {nome:<26} dia {dia:>3} (atraso de {dia - 60} dias)")

In [ ]:
# Sua vez: invente o seu mercado (queda, alta, lateral) e veja quem vira primeiro
dias = np.arange(200)
meu = np.where(dias < 80, 100 - 0.3 * dias, 76 + 0.6 * (dias - 80))
s = pd.DataFrame({"Close": meu, "High": meu + 0.5, "Low": meu - 0.5, "Volume": 1.0})
for nome, cond in {"IFR > 50": ifr(s["Close"]) > 50, "MACD > 0": macd(s["Close"]) > 0}.items():
    print(nome, "virou no dia", int(cond[cond & (cond.index > 80)].index.min()))

## Parte 8: aquecimento

In [ ]:
c = IBOV["Close"]
ALVO = c.index[-1]
AQ = {b: float(ifr(c.iloc[-b:]).loc[ALVO]) for b in (20, 30, 50, 100, 250, 1000)}
print(f"\nParte 8. IFR(14) do Ibovespa em {ALVO:%d/%m/%Y}, mudando só o tamanho do histórico:")
for b, v in AQ.items():
    print(f"   {b:>5} barras: IFR = {nr(v)}")

## Parte 9: sobreajuste

In [ ]:
def retorno_ifr(c, n, baixo, alto):
    """Compra quando o IFR cai abaixo de 'baixo', vende quando passa de 'alto' (no pregão seguinte). Retorno anual."""
    sinal = ifr(c, n).to_numpy()
    posicao, pos = np.zeros(len(c)), 0
    for i, v in enumerate(sinal):
        if pos == 0 and v < baixo:
            pos = 1
        elif pos == 1 and v > alto:
            pos = 0
        posicao[i] = pos
    ret = c.pct_change().fillna(0).to_numpy() * np.roll(posicao, 1)
    return (np.prod(1 + ret) ** (252 / len(ret)) - 1) * 100


ib = IBOV.loc["2010":, "Close"]
DENTRO, FORA = ib.loc[:"2019-12-31"], ib.loc["2020-01-01":]
GRADE = [(n, b, a) for n in (2, 3, 5, 7, 10, 14, 20, 30) for b in (10, 20, 30, 40) for a in (60, 70, 80, 90)]
RES = pd.DataFrame([(n, b, a, retorno_ifr(DENTRO, n, b, a), retorno_ifr(FORA, n, b, a)) for n, b, a in GRADE],
                   columns=["n", "baixo", "alto", "dentro", "fora"])
MELHOR = RES.loc[RES["dentro"].idxmax()]
POS_FORA = int((RES["fora"] > MELHOR["fora"]).sum()) + 1
SEGURAR = ((FORA.iloc[-1] / FORA.iloc[0]) ** (252 / len(FORA)) - 1) * 100
CORR_DF = np.corrcoef(RES["dentro"], RES["fora"])[0, 1]
print(f"\nParte 9. {len(GRADE)} combinações de IFR no Ibovespa: 2010-2019 (dentro) e 2020 em diante (fora)")
print(f"   Campeã de 2010-2019: IFR({int(MELHOR.n)}), compra abaixo de {int(MELHOR.baixo)}, vende acima de"
      f" {int(MELHOR.alto)}: {nr(MELHOR.dentro, 1)}% ao ano dentro; {nr(MELHOR.fora, 1)}% ao ano fora"
      f" ({POS_FORA}ª de {len(GRADE)} fora)")
print(f"   Mediana fora: {nr(RES['fora'].median(), 1)}% ao ano; comprar e segurar o Ibovespa fora:"
      f" {nr(SEGURAR, 1)}% ao ano; combinações que bateram o índice fora: {int((RES['fora'] > SEGURAR).sum())}")
print(f"   Correlação entre o resultado dentro e fora da amostra: {nr(CORR_DF)}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `ifr` da Parte 2 e a célula da Parte 8. Mostre por que a suavização de Wilder (`ewm(alpha=1 / n)`) guarda memória de todo o histórico e por que o IFR(14) calculado com 20 barras sai diferente do calculado com 1.000." Confira com a Parte 8, que imprime o IFR(14) do Ibovespa em 30/09/2026 com históricos de 20 a 1.000 barras: 53,87 com 20 e 60,90 com 250, os números do Gráfico 8.6.
2. **Adaptar:** "Crie uma célula nova no fim do notebook que baixe [ATIVO], com o ticker do Yahoo Finance, pelas funções `baixa` e `limpa`, e rode nele, desde 2019, a função `nove`, a correlação de postos e a função `opinioes`, como na Parte 3. Não mude as fórmulas dos indicadores nem os períodos." Antes de olhar o ativo novo, confira se a Parte 3 continua dando, na PETR4, correlação média de 0,73 e 1,6 opiniões independentes.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 3, troque PETR4 pela sua ação favorita. Quantas opiniões independentes sobram?
2. Na Parte 4, troque o OBV pelo MFI. O volume continua conversando com o momentum?
3. Na Parte 9, use 2015-2022 como dentro da amostra e 2023 em diante como fora. A campeã muda?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.